In [2]:
import pandas as pd
df = pd.read_csv('diabetes.csv')

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
# memisahkan atribut pada dataset dan menyimpannya pada sebuah variabel
X = df[df.columns[:8]]

# memisahkan label pada dataset dan menyimpannya pada sebuah variabel
y = df['Outcome']

In [6]:
from sklearn.preprocessing import StandardScaler

# standarisasi nilai-nilai dari dataset
scaler = StandardScaler()
scaler.fit(X)
X = scaler.transform(X)

In [7]:
from sklearn.model_selection import train_test_split

# memisahkan data untuk training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [8]:
from sklearn.svm import SVC

# membuat objek SVC dan memanggil fungsi fit untuk melatih model
clf = SVC()
clf.fit(X_train, y_train)

SVC()

In [9]:
# Menampilkan skor akurasi prediksi
clf.score(X_test, y_test)

0.7559055118110236

---
# Penjelasan: Identifikasi dan Kategori Dataset

## 1. Karakteristik dataset (`diabetes.csv`)

- **Ukuran:** 768 baris dan 9 kolom, tanpa nilai kosong (NaN).
- **Fitur (8 atribut numerik):** Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age.
- **Label/target:** `Outcome`, bernilai 0 (tidak diabetes) dan 1 (diabetes).
- **Distribusi kelas:**
  - Kelas 0 (non-diabetes): 500 data (65,1%)
  - Kelas 1 (diabetes): 268 data (34,9%)

## 2. Kategori dataset

| Aspek | Kategori |
|---|---|
| Jenis pembelajaran | **Supervised learning** (data berlabel) |
| Jenis tugas | **Klasifikasi biner** (2 kelas) |
| Tipe data | **Tabular / terstruktur**, fitur numerik |
| Domain | **Kesehatan / medis** (prediksi diabetes) |
| Keseimbangan kelas | **Imbalanced ringan** (rasio sekitar 65:35) |

Alasannya: kolom `Outcome` berupa label diskret (0/1), bukan nilai kontinu, sehingga ini masalah klasifikasi dan bukan regresi. Notebook ini juga memisahkan fitur (`X`) dari label (`y`), yang menandakan pembelajaran terawasi.

## 3. Penjelasan hasil kode di atas (SVM)

Alur notebook:
1. Memuat data dan memisahkan `X` (8 fitur) dari `y` (`Outcome`).
2. Standarisasi fitur dengan `StandardScaler`.
3. Membagi data: 67% training dan 33% testing (`random_state=42`).
4. Melatih `SVC()` dengan kernel default RBF.
5. Mengevaluasi dengan `clf.score()`.

**Akurasi: 0,7559 (≈ 75,6%)** pada 254 data uji.

Rincian per kelas (dari menjalankan ulang kode yang sama, karena tidak ditampilkan di notebook ini):

| Kelas | Precision | Recall | F1-score |
|---|---|---|---|
| 0 (non-diabetes) | 0,80 | 0,85 | 0,82 |
| 1 (diabetes) | 0,66 | 0,58 | 0,62 |

Confusion matrix: 142 benar diprediksi 0, 26 salah diprediksi 1, 36 salah diprediksi 0, dan 50 benar diprediksi 1.

## 4. Interpretasi

- Model cukup baik sebagai **baseline**, tetapi **lebih kuat mengenali kelas non-diabetes**.
- **Recall kelas diabetes hanya 58%**: sekitar 42% pasien diabetes (36 dari 86) tidak terdeteksi. Dalam konteks medis, kesalahan jenis ini (*false negative*) lebih berisiko.
- Penyebab yang mungkin:
  - **Ketidakseimbangan kelas**, sehingga model cenderung condong ke kelas mayoritas.
  - **Nilai 0 yang sebenarnya data hilang** pada beberapa kolom medis:
    - Insulin: 374 baris
    - SkinThickness: 227 baris
    - BloodPressure: 35 baris
    - BMI: 11 baris
    - Glucose: 5 baris
  - Hyperparameter (C, gamma) belum dituning.

### Saran perbaikan

- Ganti nilai 0 pada kolom medis dengan median (atau teknik imputasi lain).
- Gunakan `SVC(class_weight='balanced')`.
- Tuning hyperparameter dengan `GridSearchCV`.
- Evaluasi dengan cross-validation dan perhatikan recall/F1 kelas diabetes, bukan hanya akurasi.

## 5. Kesimpulan

Dataset `diabetes.csv` termasuk **klasifikasi biner** dengan pendekatan **supervised learning**, berupa **data tabular medis** dengan 8 fitur numerik dan label `Outcome` (0/1). Model SVM default mencapai **akurasi ±75,6%**, tetapi masih lemah dalam mendeteksi kelas diabetes.

In [14]:
import pandas as pd
import numpy as np
from sklearn.utils.multiclass import type_of_target

y = df['Outcome']

jenis = type_of_target(y)
print('Hasil type_of_target:', jenis)
print('Nilai unik label    :', np.unique(y))
print('Jumlah kelas        :', y.nunique())
print('Jumlah kolom label  :', 1 if y.ndim == 1 else y.shape[1])
print()

# tabel perbandingan tiga jenis klasifikasi
syarat = pd.DataFrame({
    'Jenis': ['Biner (binary)', 'Multikelas (multiclass)', 'Multilabel'],
    'Jumlah kolom label': ['1', '1', '>= 2 (matriks indikator 0/1)'],
    'Jumlah kelas': ['tepat 2', '> 2', 'setiap label bisa 0/1 bebas'],
    'Satu sampel punya': ['1 kelas', '1 kelas', 'beberapa label sekaligus'],
    'type_of_target': ['binary', 'multiclass', 'multilabel-indicator'],
})
syarat['Cocok dengan dataset?'] = np.where(syarat['type_of_target'] == jenis, 'YA', 'tidak')
syarat

Hasil type_of_target: binary
Nilai unik label    : [0 1]
Jumlah kelas        : 2
Jumlah kolom label  : 1



,Jenis,Jumlah kolom label,Jumlah kelas,Satu sampel punya,type_of_target,Cocok dengan dataset?
0,Biner (binary),1,tepat 2,1 kelas,binary,YA
1,Multikelas (multiclass),1,> 2,1 kelas,multiclass,tidak
2,Multilabel,>= 2 (matriks indikator 0/1),setiap label bisa 0/1 bebas,beberapa label sekaligus,multilabel-indicator,tidak
